# Chapter 0 — Numbers in a Machine

Every number quoted in Chapter 0, recomputed. Each cell **asserts** the claim it checks, so
`python tests/run_notebooks.py 00` fails if the chapter and reality ever disagree.

In [ ]:
import math
import torch

def derive(E, M, flavour="ieee"):
    """The five formulas of section 0.4."""
    bias = 2 ** (E - 1) - 1
    top_field = 2 ** E - 2 if flavour == "ieee" else 2 ** E - 1
    top_mant = 2 ** M - 2 if flavour == "fn" else 2 ** M - 1
    return dict(bias=bias,
                max=(1 + top_mant / 2 ** M) * 2.0 ** (top_field - bias),
                min_normal=2.0 ** (1 - bias),
                min_subnormal=2.0 ** (1 - bias - M),
                eps=2.0 ** -M)

for name, (E, M, fl, dt) in {"fp32": (8, 23, "ieee", torch.float32), "fp16": (5, 10, "ieee", torch.float16),
                             "bf16": (8, 7, "ieee", torch.bfloat16), "e4m3": (4, 3, "fn", torch.float8_e4m3fn),
                             "e5m2": (5, 2, "ieee", torch.float8_e5m2)}.items():
    d, fi = derive(E, M, fl), torch.finfo(dt)
    assert math.isclose(d["max"], fi.max, rel_tol=1e-12) and d["min_normal"] == fi.tiny and d["eps"] == fi.eps, name
    print(f"{name:5s} max={d['max']:.6g} min_normal={d['min_normal']:.6g} min_sub={d['min_subnormal']:.6g} eps={d['eps']:.6g}  == torch.finfo")
for name, (E, M) in {"fp6 e3m2": (3, 2), "fp6 e2m3": (2, 3), "fp4 e2m1": (2, 1)}.items():
    print(f"{name}: max = {derive(E, M, 'none')['max']}")
assert derive(3, 2, "none")["max"] == 28 and derive(2, 3, "none")["max"] == 7.5 and derive(2, 1, "none")["max"] == 6
assert derive(4, 3, "ieee")["max"] == 240        # what e4m3 would be with IEEE rules

## Every e4m3 and e5m2 pattern, decoded by hand and by PyTorch

In [ ]:
def decode(bits, E, M, flavour="ieee"):
    s = (bits >> (E + M)) & 1; e = (bits >> M) & ((1 << E) - 1); m = bits & ((1 << M) - 1)
    bias = (1 << (E - 1)) - 1
    if flavour == "ieee" and e == (1 << E) - 1:
        return float("nan") if m else (-1) ** s * float("inf")
    if flavour == "fn" and e == (1 << E) - 1 and m == (1 << M) - 1:
        return float("nan")
    v = (m / (1 << M)) * 2.0 ** (1 - bias) if e == 0 else (1 + m / (1 << M)) * 2.0 ** (e - bias)
    return -v if s else v

for dt, E, M, fl in [(torch.float8_e4m3fn, 4, 3, "fn"), (torch.float8_e5m2, 5, 2, "ieee")]:
    ref = torch.arange(256, dtype=torch.uint8).view(dt).float().tolist()
    for b in range(256):
        a, r = decode(b, E, M, fl), ref[b]
        assert (math.isnan(a) and math.isnan(r)) or a == r, (dt, b, a, r)
    print(dt, "all 256 patterns match")
fp4 = [decode(b, 2, 1, "none") for b in range(8)]
print("fp4 e2m1:", fp4); assert fp4 == [0, 0.5, 1, 1.5, 2, 3, 4, 6]
finite = [decode(b, 4, 3, "fn") for b in range(256)]
assert len({v for v in finite if not math.isnan(v)}) == 253   # 254 finite patterns, +0 == -0

## Integers, gaps and rounding

In [ ]:
q = lambda x, dt: torch.tensor(float(x)).to(dt).float().item()
b, h, e4 = torch.bfloat16, torch.float16, torch.float8_e4m3fn
assert (torch.tensor(100, dtype=torch.int8) + torch.tensor(100, dtype=torch.int8)).item() == -56
assert format(torch.tensor(-5, dtype=torch.int8).view(torch.uint8).item(), "08b") == "11111011"
checks = {
    "bf16 257 -> 256 (tie to even)": q(257, b) == 256, "bf16 259 -> 260": q(259, b) == 260,
    "bf16 261 -> 260": q(261, b) == 260, "fp16 2049 -> 2048": q(2049, h) == 2048,
    "fp16 1000.3 -> 1000.5": q(1000.3, h) == 1000.5, "bf16 100.3 -> 100.5": q(100.3, b) == 100.5,
    "fp32 1e8 + 1 == 1e8": (torch.tensor(1e8) + 1).item() == torch.tensor(1e8).item(),
    "fp16 4097, 4098, 4099 -> 4096, 4096, 4100": [q(x, h) for x in (4097, 4098, 4099)] == [4096, 4096, 4100],
    "bf16 0.999 == 1.0": q(0.999, b) == 1.0,
    "fp16 zero threshold is 2^-25": q(2 ** -25, h) == 0 and q(3e-8, h) == 2 ** -24 and q(2.5e-8, h) == 0,
    "fp16 1e-6 -> 1.0133e-6 (subnormal)": abs(q(1e-6, h) - 1.0132789611816406e-06) < 1e-15,
    "e4m3 300 -> 288, 17 -> 16, -2.6 -> -2.5": [q(x, e4) for x in (300, 17, -2.6)] == [288, 16, -2.5],
    "e4m3 500 saturates to 448": q(500, e4) == 448,
    "e4m3 0x7E is 448": torch.tensor(0b01111110, dtype=torch.uint8).view(e4).float().item() == 448,
}
for k, v in checks.items():
    print("ok " if v else "BAD", k); assert v, k

## The frozen Adam average (exercise 0.4)

In [ ]:
for beta2 in (0.999, 0.99, 0.95):
    for dt in (torch.float32, torch.float16, torch.bfloat16):
        v, B = torch.tensor(1.0, dtype=dt), torch.tensor(beta2, dtype=dt)
        for _ in range(1000):
            v = v * B
        print(f"beta2={beta2}  {str(dt)[6:]:9s} stored as {B.item():.8f}   v after 1000 = {v.item():.4g}   exact {beta2 ** 1000:.4g}")
v = torch.tensor(1.0, dtype=torch.bfloat16)
for _ in range(1000):
    v = v * torch.tensor(0.999, dtype=torch.bfloat16)
assert v.item() == 1.0

## Stochastic rounding: 256 + 1, a thousand times, in bf16

In [ ]:
def ulp_bf16(x):
    return 2.0 ** (math.floor(math.log2(abs(x))) - 7)
g = torch.Generator().manual_seed(0)
acc = torch.tensor(256.0, dtype=torch.bfloat16)
for _ in range(1000):
    acc = acc + torch.tensor(1.0, dtype=torch.bfloat16)
print("round to nearest:", acc.item())
assert acc.item() == 256
runs = []
for r in range(2000):
    x = 256.0
    u = torch.rand(1000, generator=g).tolist()
    for i in range(1000):
        t = x + 1; step = ulp_bf16(t); lo = math.floor(t / step) * step
        x = lo + step if u[i] < (t - lo) / step else lo
    runs.append(x)
mean = sum(runs) / len(runs); sd = (sum((r - mean) ** 2 for r in runs) / len(runs)) ** .5
print(f"stochastic rounding: mean {mean:.1f} over 2000 runs (exact 1256), one run spreads by +-{sd:.0f}")
assert abs(mean - 1256) < 6 * sd / len(runs) ** .5

## Block scaling: MX vs NVFP4 on one block, and int8

In [ ]:
FP4 = [0, .5, 1, 1.5, 2, 3, 4, 6]
def fp4(x):                          # round to nearest, ties to even mantissa, saturate at 6
    a = min(abs(x), 6.0); best = min(FP4, key=lambda v: (abs(v - a), FP4.index(v) % 2))
    return math.copysign(best, x) if best else 0.0
blk = [0.72, -1.9, 0.05, 3.3, -0.4, 0.0, 2.2, -0.9]
amax = max(map(abs, blk))
rmse = lambda d: (sum((a - b) ** 2 for a, b in zip(d, blk)) / len(blk)) ** .5
X = 2.0 ** (math.floor(math.log2(amax)) - 2)                 # MX: E8M0 scale, emax(e2m1) = 2
mx = [fp4(v / X) * X for v in blk]
s = torch.tensor(amax / 6).to(torch.float8_e4m3fn).float().item()   # NVFP4: e4m3 block scale
nv = [fp4(v / s) * s for v in blk]
print("MX     scale", X, " ->", mx, " rmse", round(rmse(mx), 4))
print("NVFP4  scale", s, " ->", [round(v, 5) for v in nv], " rmse", round(rmse(nv), 4))
assert X == 0.5 and mx[3] == 3.0 and s == 0.5625 and nv[3] == 3.375
assert round(rmse(mx), 3) == 0.143 and round(rmse(nv), 3) == 0.105

w = [0.12, -0.5, 0.33, 0.04]; sc = 0.5 / 127
qs = [max(-127, min(127, round(v / sc))) for v in w]
print("int8 symmetric:", qs, [round(q * sc, 4) for q in qs]); assert qs == [30, -127, 84, 10]
sa = 7 / 255; z = round(1 / sa)
qa = [max(0, min(255, round(v / sa) + z)) for v in (-1, 0, 2.7, 6)]
print("uint8 asymmetric: z =", z, qa); assert z == 36 and qa == [0, 36, 134, 255]